# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`  
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset package using the `mlcroissant` library. All dataset components are referenced by their unique `@id` in accordance with Croissant best practices.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata using `mlcroissant`. We will also preview the dataset description for context.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (use .metadata, not subscript notation)
metadata_dict = dataset.metadata.to_json()
print(f"{metadata_dict['name']}: {metadata_dict['description']}")
print(f"Dataset identifier: {metadata_dict.get('identifier','')}")
print(f"Published: {metadata_dict.get('datePublished','')}")

## 2. Data Overview
Inspect available record sets and fields. All references are by their Croissant `@id`.

First, let's list all available `@id` for record sets in the Croissant metadata.

In [ ]:
# List all record sets using their @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the metadata.")
else:
    print(f"Number of record sets: {len(record_sets)}")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}, Name: {rs.get('name','<none>')}")

# For demonstration, let's print the fields and their @id for the first record set, if available
if record_sets:
    first_rs = record_sets[0]
    print(f"\nFields in RecordSet '{first_rs['@id']}' ({first_rs.get('name','')}):")
    for field in first_rs.get('field', []):
        # Each field is typically an object; get @id and name
        if isinstance(field, dict):
            print(f"  - Field @id: {field.get('@id','')}, Name: {field.get('name','')}, DataType: {field.get('dataType','')}")
        else:
            print(f"  - Field reference: {field}")

### Examine one sample record from the primary record set

Assume the main data table is the first listed record set—let's sample a record and print the keys, referencing fields by their `@id`.

In [ ]:
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"\nSampling a record from RecordSet @id: {main_record_set_id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
            print(json.dumps(rec, indent=2))
            if i == 0:
                break
        print(f"\nField @ids in the sample record: {list(rec.keys())}")
    except Exception as e:
        print(f"Error retrieving records: {e}")
else:
    print("No record sets available for record sampling.")

## 3. Data Extraction
Load data from all record sets into Pandas DataFrames for analysis. `@id` values are used for references.

We will dynamically list the found record set `@id`s and load each as a DataFrame.

In [ ]:
# Extract all listed record sets (by @id) into DataFrames
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []
dataframes = {}

for rs_id in record_set_ids:
    # Load all records for this record set
    rs_records = list(dataset.records(record_set=rs_id))
    if rs_records:
        df = pd.DataFrame(rs_records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {rs_id}")
    else:
        print(f"No records for RecordSet @id: {rs_id}")

if dataframes:
    # Show the columns of the main record set (first one)
    main_df_id = record_set_ids[0]
    print("\nFields (@id) in main DataFrame:")
    print(list(dataframes[main_df_id].columns))
    dataframes[main_df_id].head()
else:
    print("No dataframes were loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's examine data distributions, filter records, normalize numeric fields, and perform some basic grouping. All field accesses use Croissant `@id`.

First, we print all field `@id`s in the primary DataFrame so you can select a numeric and a group field for further analysis.

In [ ]:
# Show all available column @id for the main dataset
if dataframes:
    main_df = dataframes[main_df_id]
    print("Field @ids in the main record set:")
    print(list(main_df.columns))

    # Pick an example numeric field (by inspecting the columns)
    # For demonstration, search for a likely numeric field
    numeric_candidates = [col for col in main_df.columns if any(sub in col.lower() for sub in ['age','interval','years','months','count','number','size'])]
    print("\nDetected numeric field candidates:", numeric_candidates)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field for EDA: '{numeric_field}'")
    else:
        print("No numeric field found for demonstration. Please adjust field selection manually.")

    # Look for a possible group field (@id) such as a categorical/label field
    group_candidates = [col for col in main_df.columns if any(grp in col.lower() for grp in ['sex','site','status','histology','msi','anatomical','stage'])]
    print("Group-able field candidates:", group_candidates)
    group_field = group_candidates[0] if group_candidates else None
else:
    numeric_field = None
    main_df = None
    group_field = None
    print("No main DataFrame loaded for EDA.")

Now, let's filter the data on the numeric field, normalize it, and group by the group-able field.

In [ ]:
if main_df is not None and numeric_field:
    # Convert to numeric (if not already)
    main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
    threshold = main_df[numeric_field].mean() if main_df[numeric_field].notnull().any() else 0
    print(f"Applying a filter: {numeric_field} > {threshold}")
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered DataFrame shape: {filtered_df.shape}")

    # Normalize the numeric field
    norm_col = numeric_field + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nFirst rows of normalized {numeric_field}:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Group by the group_field if available
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
        print(grouped_df)
else:
    print("No numeric field found for EDA or DataFrame not loaded.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and show group means as a bar chart (if a group field was detected).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field:
    plt.figure(figsize=(6, 4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field and group_field in main_df.columns:
        plt.figure(figsize=(8, 4))
        sns.barplot(data=main_df, x=group_field, y=numeric_field, ci=None)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR^2 clinical colorectal cancer dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library. Key steps included:
- Loading Croissant metadata and data records via `@id` references
- Inspecting available record sets and fields programmatically
- Loading all record sets as DataFrames and exploring their structure
- Performing basic filtering, normalization, and group analysis using field `@id`s
- Visualizing data distributions and group comparisons

This approach ensures rigorous and reproducible access to both metadata and content as required by FAIR principles. For further analysis, refer to entity `@id` values for field selection, merging, or advanced modeling.